# NOTEBOOK FOR PREPROCESSING LINE DATA

### IMPORTS

In [ ]:
import shapefile
import geopandas as gpd
import numpy as np

from matplotlib.cm import get_cmap

from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import splprep, splev
from shapely.geometry import LineString
from scipy.interpolate import interp1d

import torch
from torch_geometric.data import Data
import matplotlib.pyplot as plt
from shapely.geometry import shape as shapely_shape
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter1d

In [ ]:
# PLOTTING FUNCTION 

def plot_lines(lines, lines_syn_1=None, lines_syn_2=None, start=0, end=100, figsize=(10, 6), title=None):

    plt.figure(figsize=figsize)

    for i in range(start,end):
        line = lines[i]
        x, y = line[:, 0], line[:, 1]
        #plt.scatter(x, y, color='#00305D', label='Original Line', s=20)
        plt.plot(x, y, color='#00305D', label=f'Original Line')

        if lines_syn_1 is not None:
            line_s1 = lines_syn_1[i]
            x_s1, y_s1 = line_s1[:, 0], line_s1[:, 1]
            plt.plot(x_s1, y_s1, color='#EC9A29', label=f'Synthetic Original')

        if lines_syn_2 is not None:
            line_s2 = lines_syn_2[i]
            x_s2, y_s2 = line_s2[:, 0], line_s2[:, 1]
            plt.plot(x_s2, y_s2, color='#A8201A', label=f'Synthetic Displaced')

        if title:
            plt.title(f'{title} - Line number {i}')
        else:
            plt.title(f'Line number {i}')
        
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.axis('equal')
        plt.grid(True)
        plt.legend()
        plt.show()

## Load Input Shapefile

In [ ]:
PATH_SHP = '../data/original/waterways_merged_reprojected.shp'

In [ ]:
def read_line_shp(path, num_points):

  with shapefile.Reader(path) as sf:
    shapes = sf.shapes() 
    print("Original Shapes", len(shapes))
    sublines = []
    for shape in shapes: 
      num_lines = len(shape.points) // num_points  
      
      i = num_points
      for _ in range(num_lines): 
        subline_points = shape.points[i-num_points : i]
        sublines.append(subline_points) 
        i += num_points

  arr = np.array(sublines)
  print("Shape of original array", arr.shape)

  return arr

In [ ]:
# segment_length = 128
# original_lines = read_line_shp(PATH_SHP, num_points=segment_length)

## Fixed Length Segments

In [ ]:
def interpolate_line(points, num_points):

    points = np.array(points)
    cumulative_distance = np.cumsum(np.sqrt(np.sum(np.diff(points, axis=0)**2, axis=1)))
    cumulative_distance = np.insert(cumulative_distance, 0, 0)

    f_interp = interp1d(cumulative_distance, points, axis=0, kind='linear')
    new_distances = np.linspace(0, cumulative_distance[-1], num_points)
    return f_interp(new_distances)

def read_line_shp_interpolated(path, num_points=64, min_points=32, max_points=256):

    with shapefile.Reader(path) as sf:
        shapes = sf.shapes()
        sublines = []
        shapes = [shape for shape in shapes if max_points>= len(shape.points) >= min_points]

        for shape in shapes:
            pts = shape.points
            total_len = len(pts)

            num_full_segments = total_len // min_points
            remainder = total_len % min_points

            for i in range(num_full_segments):
                segment = pts[i*min_points : (i+1)*min_points]
                interpolated = interpolate_line(segment, num_points)
                sublines.append(interpolated)

            if remainder >= min_points:
                leftover = pts[-remainder:]
                interpolated = interpolate_line(leftover, num_points)
                sublines.append(interpolated)

            elif remainder > 0:
                padded_segment = pts[-(min_points):]
                interpolated = interpolate_line(padded_segment, num_points)
                sublines.append(interpolated)

    arr = np.array(sublines)
    print(f"Final shape: {arr.shape}  (n_lines, num_points, 2)")
    return arr

In [ ]:
# Interpolated Lines 
#original_lines = read_line_shp_interpolated(PATH_SHP, num_points=64, min_points=32, max_points=256)

## **Rotation**

In [ ]:
def rotate_line_to_vertical_up(points):

    p0 = points[0]
    p1 = points[-1]

    translated = points - p0
    vec = p1 - p0

    angle = np.arctan2(vec[1], vec[0])  
    rotation_angle = np.pi/2 - angle    

    cos_a = np.cos(rotation_angle)
    sin_a = np.sin(rotation_angle)
    R = np.array([[cos_a, -sin_a],
                  [sin_a,  cos_a]])

    rotated = translated @ R.T

    if rotated[-1, 1] < 0:
        rotated[:, 1] *= -1

    return rotated

In [ ]:
# Rotation with interpolated lines 
#rotated_lines = np.array([rotate_line_to_vertical_up(line) for line in original_lines])

# Roation with original shapefile lines 
# rotated_lines = [rotate_line_to_vertical_up(coords) for coords in coords_list]

# plot_lines(rotated_lines, None, None, start=start, end=start+1, title="Rotated Lines")
# np.save(f'../data/preprocessing/rotated/rotated_lines.npy', rotated_lines)

## Creation Synthetic Lines

In [ ]:
interpolated_lines = np.load(f'../data/preprocessing/interpolated/interpolated_lines.npy')

In [ ]:
def resample_line(line, n_points):
    distances = np.cumsum(np.linalg.norm(np.diff(line, axis=0), axis=1))
    distances = np.insert(distances, 0, 0)
    total_length = distances[-1]
    uniform_distances = np.linspace(0, total_length, n_points)
    interp_func = interp1d(distances, line, axis=0)
    return interp_func(uniform_distances)

In [ ]:
def chaikin_smoothing(points, iterations=2):
    for _ in range(iterations):
        new_points = [points[0]]
        for i in range(len(points) - 1):
            p0, p1 = points[i], points[i + 1]
            Q = 0.75 * p0 + 0.25 * p1
            R = 0.25 * p0 + 0.75 * p1
            new_points.extend([Q, R])
        new_points.append(points[-1])
        points = np.array(new_points)
    return points

In [ ]:
def generate_offset_line(line, offset_distance, noise_std=10, smoothing_iters=3,
                         adaptive_scaling=True, curvature_sigma=8):
    n_points = len(line)

    # Compute tangent vectors
    dx = gaussian_filter1d(np.gradient(line[:, 0]), sigma=curvature_sigma)
    dy = gaussian_filter1d(np.gradient(line[:, 1]), sigma=curvature_sigma)

    norms = np.sqrt(dx**2 + dy**2) + 1e-8
    dx /= norms
    dy /= norms

    # Compute normal vectors
    nx = -dy
    ny = dx

    # Offset magnitudes
    offsets = offset_distance + np.random.normal(0, noise_std, size=n_points)

    if adaptive_scaling:
        angles = np.arctan2(dy, dx)
        curvature = np.abs(np.gradient(angles))
        curvature = gaussian_filter1d(curvature, sigma=3)
        curvature_factor = 1 / (1 + curvature * 50) 
        offsets *= curvature_factor

    # Apply offsets
    offset_x = nx * offsets
    offset_y = ny * offsets
    offset_line = line + np.stack([offset_x, offset_y], axis=1)

    # Optional: Smoothing
    for _ in range(smoothing_iters):
        offset_line = chaikin_smoothing(offset_line)

    return offset_line

In [ ]:
def is_valid_line(line, river, min_distance=5.0, loop_area_threshold=300.0, simplify_tolerance=5.0, id=0):

    line_geom = LineString(line)
    river_geom = LineString(river)

    # Reject if the line intersects itself
    if not line_geom.is_simple:
        return False


    # Reject if simplified version is still too dense (indicating loops)
    simplified = line_geom.simplify(simplify_tolerance)
    if len(simplified.coords) > 2 * len(line):
        return False

    # Reject if area enclosed is unusually small (tight loops)
    area = line_geom.buffer(1.0).area
    if area < loop_area_threshold:
        return False

    return True

In [ ]:
def generate_synthetic_lines(
    rivers, n_points=64, close_offset=40,
    min_distance_far=100.0, push_strength=0.4, max_attempts=32
):

    if isinstance(rivers, np.ndarray):
        n_lines = rivers.shape[0]
        get_line = lambda i: rivers[i]
    elif isinstance(rivers, (list, tuple)):
        n_lines = len(rivers)
        get_line = lambda i: np.asarray(rivers[i])
    else:
        raise TypeError("Input 'rivers' must be a NumPy array or list of NumPy arrays.")

    close_lines = []
    far_lines = []

    invalid_close_ids = []
    invalid_far_ids = []

    for i in range(n_lines):
        river = resample_line(get_line(i), n_points)

        # --- Generate close line ---
        close_valid = False
        for attempt in range(max_attempts):
            close_candidate = generate_offset_line(river, offset_distance=close_offset, noise_std=10)
            if is_valid_line(close_candidate, river, min_distance=4.0, id=i):
                close_line = resample_line(close_candidate, n_points)
                close_valid = True
                break
        else:
            close_line = resample_line(close_candidate, n_points)

        if not close_valid:
            invalid_close_ids.append(i)

        close_lines.append(close_line)

        # --- Generate far line (ground truth) ---
        b_copy = close_line.copy()
        tree_river = cKDTree(river)
        distances, indices = tree_river.query(b_copy, k=1)
        nearest_river = river[indices]

        deficit = min_distance_far - distances
        mask = deficit > 0

        push_vectors = b_copy - nearest_river
        norms = np.linalg.norm(push_vectors, axis=1) + 1e-8
        push_vectors /= norms[:, None]
        push_vectors *= deficit[:, None] * push_strength

        push_vectors[:, 0] = gaussian_filter1d(push_vectors[:, 0], sigma=2)
        push_vectors[:, 1] = gaussian_filter1d(push_vectors[:, 1], sigma=2)

        b_copy[mask] += push_vectors[mask]
        
        if not is_valid_line(b_copy, river, min_distance=min_distance_far, id=i):
            invalid_far_ids.append(i)

        far_lines.append(b_copy)

    close_lines = np.stack(close_lines)
    far_lines = np.stack(far_lines)

    return close_lines, far_lines, np.array(invalid_close_ids), np.array(invalid_far_ids)


In [ ]:
close, far, idc, idf = generate_synthetic_lines(
    interpolated_lines,
    n_points=64,
    close_offset=300,
    min_distance_far=200,
    push_strength=0.6, 
    max_attempts=32
)

In [ ]:
# FILTERING INVALID LINES 

# Merge all bad indices
bad_ids = np.unique(np.concatenate([idc, idf]))
print(len(bad_ids))

# Create mask (True = keep, False = remove)
mask = np.ones(len(interpolated_lines), dtype=bool)
mask[bad_ids] = False

# Apply to all aligned arrays
rivers_clean = interpolated_lines[mask]
close_clean  = close[mask]
far_clean    = far[mask]
print(len(rivers_clean), len(close_clean),len(far_clean))

In [ ]:
plot_lines(rivers_clean, close_clean, far_clean, start=0, end=5, title="Original and Synthetic Lines")

In [ ]:
# np.save(f'../data/preprocessing/interpolated/interpolated_newsyn_original', rivers_clean)
# np.save(f'../data/preprocessing/interpolated/interpolated_newsyn_close.npy', close_clean)
# np.save(f'../data/preprocessing/interpolated/interpolated_newsyn_far.npy', far_clean)

## Min Max Normalization

In [ ]:
# rotated_lines_close = np.load(f'../data/preprocessing/rotated/rotated_lines_newsyn_close.npy')
# rotated_lines_far = np.load(f'../data/preprocessing/rotated/rotated_lines_newsyn_far.npy')
# rotated_lines_original = np.load(f'../data/preprocessing/rotated/rotated_lines_newsyn_original.npy')

In [ ]:
# LOCAL NORMALIZATION
def normalize_three_lines(original, close, far, method='max', range_mode='0,1'):

    assert original.shape == close.shape == far.shape
    N = original.shape[0]

    orig_norm = np.empty_like(original, dtype=np.float32)
    close_norm = np.empty_like(close, dtype=np.float32)
    far_norm   = np.empty_like(far, dtype=np.float32)

    centroids = np.zeros((N, 2), dtype=np.float32)
    scales = np.zeros((N,), dtype=np.float32)

    for i in range(N):
        stacked = np.vstack([original[i], close[i], far[i]])  # (192, 2)
        centroid = stacked.mean(axis=0)                       # (2,)
        centered = stacked - centroid                        # (192, 2)

        if method == 'max':
            # scale = max L2 distance of any point from centroid
            dists = np.linalg.norm(centered, axis=1)
            scale = dists.max()
        elif method == 'std':
            # scalar std across both dims
            scale = centered.std()
        else:
            raise ValueError("method must be 'max' or 'std'")

        if scale == 0 or np.isnan(scale):
            scale = 1.0
        
        # base normalization in [-1, 1] space
        o = (original[i] - centroid) / scale
        c = (close[i]    - centroid) / scale
        f = (far[i]      - centroid) / scale

        if range_mode == '0,1':
            o = (o + 1.0) * 0.5
            c = (c + 1.0) * 0.5
            f = (f + 1.0) * 0.5
        
        elif range_mode != '-1,1':
            raise ValueError("range_mode must be '-1,1' or '0,1'")

        orig_norm[i]  = o
        close_norm[i] = c
        far_norm[i]   = f

        centroids[i] = centroid
        scales[i]    = scale

    return orig_norm, close_norm, far_norm, centroids, scales

In [ ]:
normalized_original, normalized_synthetic_1, normalized_synthetic_2, _, _ = normalize_three_lines(rotated_lines_original, rotated_lines_close, rotated_lines_far)

In [ ]:
# GLOBAL INTERPOLATION
combined = np.vstack(interpolated_lines_original)
global_min = combined.min(axis=(0, 1))
global_max = combined.max(axis=(0, 1))

original = np.array(interpolated_lines_original)
close    = np.array(interpolated_lines_close)
far      = np.array(interpolated_lines_far)

normalized_original = (original - global_min) / (global_max - global_min)
normalized_synthetic_1 = (close - global_min) / (global_max - global_min)
normalized_synthetic_2 = (far - global_min) / (global_max - global_min)

In [ ]:
for i in range(50):
    start=i
    plot_lines(normalized_original, normalized_synthetic_1, normalized_synthetic_2, start=start, end=start+1, title="Normalized Lines")

## Save Results

In [ ]:
np.save(f'../data/preprocessing/normalized/normalized_local_norotation_new_syn_original.npy', normalized_original)
np.save(f'../data/preprocessing/normalized/normalized_local_norotation_new_syn_close.npy', normalized_synthetic_1)
np.save(f'../data/preprocessing/normalized/normalized_local_norotation_new_synl_far.npy', normalized_synthetic_2)

In [ ]:
np.save(f'../data/preprocessing/normalized/normalized_local_new_syn2_original.npy', normalized_original)
np.save(f'../data/preprocessing/normalized/normalized_local_new_syn2_close.npy', normalized_synthetic_1)
np.save(f'../data/preprocessing/normalized/normalized_local_new_syn2_far.npy', normalized_synthetic_2)

In [ ]:
# interpolated_lines_original = np.load(f'../data/preprocessing/normalized/normalized_local_original.npy')
# interpolated_lines_close = np.load(f'../data/preprocessing/normalized/normalized_local_close.npy')
# interpolated_lines_far = np.load(f'../data/preprocessing/normalized/normalized_local_far.npy')

np.save(f'../data/preprocessing/normalized/normalized_global_norotation_close.npy', normalized_synthetic_1)
np.save(f'../data/preprocessing/normalized/normalized_global_norotation_far.npy', normalized_synthetic_2)
np.save(f'../data/preprocessing/normalized/normalized_global_norotation_original.npy', normalized_original)

## REAL WORLD DATASET

In [ ]:
# Load your shapefile
shp_path = "/Users/pia/Documents/Cartography/Masterarbeit/GIS_Project/push/selected_2.shp"
gdf = gpd.read_file(shp_path)

# Resample a LineString to N points
def resample_line(line, n_points=64):
    if not isinstance(line, LineString):
        raise ValueError("Geometry must be a LineString")
    distances = np.linspace(0, line.length, n_points)
    points = [line.interpolate(d) for d in distances]
    return np.array([[p.x, p.y] for p in points])

# Create arrays for each line_type (0.0, 1.0, 2.0)
array_0 = np.array([resample_line(row.geometry) for _, row in gdf[gdf['line_type'] == 0.0].iterrows()])
array_1 = np.array([resample_line(row.geometry) for _, row in gdf[gdf['line_type'] == 1.0].iterrows()])
array_2 = np.array([resample_line(row.geometry) for _, row in gdf[gdf['line_type'] == 2.0].iterrows()])

# Check shapes
print("Line type 0 shape:", array_0.shape)
print("Line type 1 shape:", array_1.shape)
print("Line type 2 shape:", array_2.shape)


In [ ]:
def plot_lines_arrays(array_0, array_1=None, array_2=None, line_idx=0, figsize=(10,6), title=None):

    plt.figure(figsize=figsize)

    # Plot line type 0
    if array_0 is not None and line_idx < array_0.shape[0]:
        x, y = array_0[line_idx,:,0], array_0[line_idx,:,1]
        plt.plot(x, y, color='#00305D', marker='o', markersize=4, label='Original (0.0)')
    
    # Plot line type 1
    if array_1 is not None and line_idx < array_1.shape[0]:
        x, y = array_1[line_idx,:,0], array_1[line_idx,:,1]
        plt.plot(x, y, color='#EC9A29', marker='x', markersize=4, label='Synthetic Original (1.0)')
    
    # Plot line type 2
    if array_2 is not None and line_idx < array_2.shape[0]:
        x, y = array_2[line_idx,:,0], array_2[line_idx,:,1]
        plt.plot(x, y, color='#A8201A', marker='^', markersize=4, label='Synthetic Displaced (2.0)')
    
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.axis('equal')
    plt.grid(True)
    plt.title(title if title else f'Line index {line_idx}')
    plt.legend()
    plt.show()


In [ ]:
norm_push_0, norm_push_1, norm_push_2, _, _  = normalize_three_lines(array_0, array_1, array_2)

In [ ]:
plot_lines(norm_push_0, norm_push_1, norm_push_2, 0, 1, title='Push Data')

In [ ]:
np.save(f'../data/final_dataset/sequence/test_dataset/push_siamese_2_original.npy', norm_push_0)
np.save(f'../data/final_dataset/sequence/test_dataset/push_siamese_2_close.npy', norm_push_1)
np.save(f'../data/final_dataset/sequence/test_dataset/push_siamese_2_far.npy', norm_push_2)